# Deep Agents: Practice Exercise

Build a deep agent that serves as a **research assistant** with both web search and Wikipedia capabilities. Your agent will use planning tools, file system access, and multiple search tools to research topics and save structured findings.

**What you'll implement:**
- Write a detailed system prompt with tool usage guidelines and workflows
- Instantiate the LLM model (ChatOpenAI)
- Create two search tools: Tavily (web search) and Wikipedia
- Create the deep agent with your model and tools

**What's provided for you:**
- Environment setup and API key loading
- File system workspace configuration
- Helper function to run and display agent responses

**Estimated time:** 20 minutes

## Setup

Run this cell to import all required libraries and configure the environment.

In [1]:
# Setup - run this cell first

import os
from pathlib import Path
from dotenv import load_dotenv
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain_openai import ChatOpenAI
from tavily import TavilyClient
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# Load environment variables
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables")
if not os.getenv("TAVILY_API_KEY"):
    raise ValueError("TAVILY_API_KEY not found in environment variables")

# Initialize Tavily client (you'll use this in your tool)
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

print("Setup complete!")

Setup complete!


## File System Workspace (Provided)

Deep agents need persistent storage for saving research findings. This workspace is already configured for you.

In [2]:
# Create workspace directory for agent files (provided)
agent_workspace = Path("./research_workspace")
agent_workspace.mkdir(exist_ok=True)

print(f"Agent workspace created at: {agent_workspace.absolute()}")

Agent workspace created at: d:\Agent\AI_Agents_Supplemental_Content\AI_Agents_Supplemental_Content\Level_3\research_workspace


## Context

You are building a **research assistant** that combines:
- **Web search** (Tavily): For current events, recent news, and up-to-date information
- **Wikipedia**: For established facts, historical information, and encyclopedic knowledge

Having both tools allows the agent to cross-reference information and provide more comprehensive research.

**Your implementation tasks:**

1. **System Prompt**: Write a detailed prompt that guides the agent's behavior
2. **LLM Model**: Instantiate a ChatOpenAI model with GPT-4o
3. **Tavily Tool**: Create a web search function using the provided `tavily_client`
4. **Wikipedia Tool**: Create a Wikipedia search tool using `WikipediaQueryRun`
5. **Deep Agent**: Use `create_deep_agent()` to create the agent with your model and tools

**Remember the four characteristics of deep agents:**
1. Detailed system prompts with workflows and examples
2. Planning tools (like `write_todos`) for task organization
3. Sub-agents for task decomposition
4. File system access for persistent memory

## Part 1: Write the System Prompt

Create a detailed system prompt that guides the agent's behavior as a research assistant. A good deep agent prompt should include:

- **Role definition**: What the agent is and what it does
- **Tool usage guidelines**: When to use each tool (`internet_search` vs `wikipedia`)
- **Workflow**: Steps for handling complex research tasks (use `write_todos` for planning)
- **Error handling**: What to do if searches fail or return conflicting information

In [14]:
def get_system_prompt() -> str:
    """
    Create a detailed system prompt for a research assistant deep agent.
    
    The prompt should include:
    - Role definition: Expert research assistant with web search and Wikipedia access
    - Tool usage guidelines:
      * internet_search: for current events, recent news, trending topics
      * wikipedia: for historical facts, scientific concepts, established knowledge
    - Workflow for complex tasks:
      1. Use write_todos to plan subtasks
      2. Use appropriate search tools
      3. Save findings to files
      4. Synthesize results
    - Error handling: retry with different keywords, note conflicting sources
    
    Returns:
        str: A comprehensive system prompt string
    
    Hint: Structure your prompt with clear sections like YOUR ROLE, TOOL USAGE,
    WORKFLOW, and ERROR HANDLING. Be specific about when to use each tool.
    """
    # TODO: Write and return a detailed system prompt
    # Include sections for: role, tool usage, workflow, error handling
    system_prompt = """You are researh assistant with access to websearch and wikipedia

    YOUR ROLE: 
    - Conduct through research on user queries
    - Break down complex queries into manageable subtasks
    - Synthesize the information  from multiple sources
    - Save the important findings to files for reference

    TOOL USAGE GUIDELINES:
    - Use internet_search for current information and facts 
    - Use wikipedia for historical facts, scientific concepts, established knowledge
    - Always cite sources when presenting information
    - Save detailed research findings to files for later reference      

    WORKFLOW:
    1. Break the task into subtasks using the write_todos tool
    2. Execute each subtask systematically
    3. Save intermediate findings to files
    4. Synthesize results into a comprehensive answer

    ERROR HANDLING:
    - If a search returns not results, try alternative keywords
    - If information conflicts between source, note the discrepancy 
    - Always verify facts from multiple sources when possible
    """
    return system_prompt


# Get the system prompt
system_prompt = get_system_prompt()

# Verify the prompt
if system_prompt:
    print("System prompt configured!")
    print(f"Prompt length: {len(system_prompt)} characters")
    print("\nFirst 200 characters:")
    print(system_prompt[:200] + "...")
else:
    print("TODO: Implement get_system_prompt()")

System prompt configured!
Prompt length: 1039 characters

First 200 characters:
You are researh assistant with access to websearch and wikipedia

    YOUR ROLE: 
    - Conduct through research on user queries
    - Break down complex queries into manageable subtasks
    - Synthes...


## Part 2: Instantiate the LLM Model

Create a ChatOpenAI model instance that will power your deep agent.

In [15]:
# TODO: Create a ChatOpenAI model instance
#
# Configure the model with:
# - model: "gpt-4o"
# - temperature: 0.1 (for consistent, factual responses)
# - api_key: Use os.getenv("OPENAI_API_KEY")
#
# Hint: model = ChatOpenAI(...)

model = ChatOpenAI(model="gpt-4o-mini",temperature=0)  # Replace with your implementation

# Verify the model is configured
if model:
    print(f"Model initialized: {model.model_name}")
else:
    print("TODO: Create the model instance")

Model initialized: gpt-4o-mini


## Part 3: Create the Tavily Web Search Tool

Create a function that wraps the Tavily API to search the web for current information.

In [16]:
def internet_search(query: str, max_results: int = 5) -> dict:
    """
    Search the web for information using Tavily.
    
    Args:
        query: The search query string
        max_results: Maximum number of results to return (default: 5)
    
    Returns:
        dict: Search results with titles, URLs, and content.
              On error, returns {"error": "error message"}.
    
    Hint: Use the global `tavily_client` and call its search() method.
    Wrap the call in a try/except to handle potential API errors.
    """
    # TODO: Implement the search function
    # 1. Use tavily_client.search(query, max_results=max_results)
    # 2. Return the results
    # 3. Handle exceptions by returning {"error": f"Search failed: {str(e)}"}
    tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    try:
        results = tavily_client.search(query, max_results=max_results)
        return results
    except Exception as e:
        return {"error": f"Search failed: {str(e)}"}
    


# Test the Tavily tool
if internet_search("test") is not None:
    print("Tavily search tool configured successfully!")
else:
    print("TODO: Implement the internet_search function")

Tavily search tool configured successfully!


## Part 4: Create the Wikipedia Search Tool

Create a Wikipedia search tool using `WikipediaQueryRun` and `WikipediaAPIWrapper`.

In [17]:
def create_wikipedia_tool():
    """
    Create a Wikipedia search tool for querying encyclopedic information.
    
    The tool should be configured with:
    - top_k_results: 3 (return top 3 Wikipedia articles)
    - doc_content_chars_max: 2000 (limit content length per result)
    
    Returns:
        WikipediaQueryRun: A configured Wikipedia search tool instance
    
    Hint: 
    1. Create a WikipediaAPIWrapper with the configuration options
    2. Pass the wrapper to WikipediaQueryRun as the api_wrapper parameter
    """
    # TODO: Implement the Wikipedia tool
    # 1. Create WikipediaAPIWrapper with top_k_results=3, doc_content_chars_max=2000
    # 2. Create and return WikipediaQueryRun with the api_wrapper
    wrapper = WikipediaAPIWrapper(top_k_results=3, doc_content_chars_max=2000)
    wikipedia_tool = WikipediaQueryRun(api_wrapper=wrapper)
    return wikipedia_tool


# Create the Wikipedia tool
wikipedia_tool = create_wikipedia_tool()

# Test the Wikipedia tool
if wikipedia_tool:
    print(f"Wikipedia tool created: {wikipedia_tool.name}")
    print(f"Description: {wikipedia_tool.description[:80]}...")
else:
    print("TODO: Implement the create_wikipedia_tool function")

Wikipedia tool created: wikipedia
Description: A wrapper around Wikipedia. Useful for when you need to answer general questions...


## Part 5: Create the Deep Agent

Use `create_deep_agent()` to create your research assistant with both tools.

In [18]:
def create_research_agent(model, tools, system_prompt, workspace):
    """
    Create a deep agent configured as a research assistant.
    
    Args:
        model: The ChatOpenAI model instance
        tools: List of tools (internet_search function and wikipedia_tool)
        system_prompt: The system prompt string
        workspace: Path to the workspace directory
    
    Returns:
        The configured deep agent instance
    
    Hint: Use create_deep_agent() with these parameters:
    - model: Your ChatOpenAI instance
    - tools: List of your tools
    - system_prompt: Your system prompt
    - backend: FilesystemBackend(root_dir=str(workspace), virtual_mode=True)
    """
    # TODO: Create and return the deep agent
    # Use create_deep_agent() with the parameters described above
     
    return create_deep_agent(
        model = model,
        tools = tools,
        system_prompt = system_prompt,
        backend = FilesystemBackend(root_dir=str(agent_workspace),
        virtual_mode=True)
    )
    

# Create the research agent with both tools
if model and system_prompt and internet_search("test") is not None and wikipedia_tool:
    research_agent = create_research_agent(
        model=model,
        tools=[internet_search, wikipedia_tool],
        system_prompt=system_prompt,
        workspace=agent_workspace
    )
    if research_agent:
        print("Research agent created successfully!")
        print("\nAgent capabilities:")
        print("- Planning tool (write_todos)")
        print("- File system tools (read, write, edit files)")
        print("- Web search via Tavily (internet_search)")
        print("- Wikipedia search (wikipedia)")
        print("- Subagent spawning (task tool)")
        print(f"- Files will be saved to: {agent_workspace.absolute()}")
    else:
        print("TODO: Implement the create_research_agent function")
else:
    print("Complete Parts 1-4 first before creating the agent")

Research agent created successfully!

Agent capabilities:
- Planning tool (write_todos)
- File system tools (read, write, edit files)
- Web search via Tavily (internet_search)
- Wikipedia search (wikipedia)
- Subagent spawning (task tool)
- Files will be saved to: d:\Agent\AI_Agents_Supplemental_Content\AI_Agents_Supplemental_Content\Level_3\research_workspace


## Helper Function (Provided)

This helper function streams and displays agent responses with clean formatting.

In [19]:
# Helper function to display agent responses (provided for you)
def stream_agent_response(agent, query, show_details=False, max_content_length=1000):
    """
    Stream and display agent responses with clean formatting.
    
    Args:
        agent: The deep agent instance
        query: The query string to send to the agent
        show_details: If True, show detailed tool arguments and planning
        max_content_length: Maximum characters to show for content (None for full content)
    """
    print(f"Query: {query if len(query) <= 100 else query[:100] + '...'}")
    print("\n" + "="*80)
    print("Streaming Agent Actions:")
    print("="*80 + "\n")
    
    seen_message_ids = set()
    
    for chunk in agent.stream(
        {"messages": [{"role": "user", "content": query}]},
        stream_mode="values"
    ):
        if "messages" in chunk:
            for message in chunk["messages"]:
                msg_id = message.id
                
                if msg_id in seen_message_ids:
                    continue
                    
                seen_message_ids.add(msg_id)
                msg_type = getattr(message, 'type', 'unknown')
                
                if msg_type == 'human':
                    content = message.content
                    if max_content_length and len(content) > max_content_length:
                        content = content[:max_content_length] + "..."
                    print(f"[USER]: {content}")
                    print("-" * 80)
                    
                elif msg_type == 'ai':
                    if hasattr(message, 'tool_calls') and message.tool_calls:
                        print(f"[AGENT] Calling {len(message.tool_calls)} tool(s):")
                        for tc in message.tool_calls:
                            tool_name = tc.get('name', 'unknown')
                            tool_args = tc.get('args', {})
                            
                            if tool_name == 'write_todos' and show_details:
                                todos = tool_args.get('todos', [])
                                print(f"  - {tool_name}: Planning {len(todos)} tasks")
                                for i, todo in enumerate(todos, 1):
                                    status = todo.get('status', 'pending')
                                    content = todo.get('content', '')[:50]
                                    print(f"      {i}. [{status}] {content}...")
                            elif tool_name == 'internet_search' and show_details:
                                query_str = tool_args.get('query', '')
                                print(f"  - {tool_name}: '{query_str}'")
                            elif tool_name == 'wikipedia' and show_details:
                                query_str = tool_args.get('query', '')
                                print(f"  - {tool_name}: '{query_str}'")
                            elif tool_name == 'write_file' and show_details:
                                path = tool_args.get('path', '')
                                print(f"  - {tool_name}: {path}")
                            else:
                                args_str = ', '.join(f'{k}={v}' for k, v in list(tool_args.items())[:2])
                                print(f"  - {tool_name}({args_str})")
                                
                    elif message.content:
                        content = message.content
                        if max_content_length and len(content) > max_content_length:
                            content = content[:max_content_length] + "..."
                        print(f"[AGENT]: {content}")
                    print("-" * 80)
                    
                elif msg_type == 'tool':
                    tool_name = getattr(message, 'name', 'unknown')
                    print(f"[TOOL: {tool_name}] Completed")
                    print("-" * 80)
    
    print("\n" + "="*80)
    print("Execution completed!")
    print("="*80)

print("Helper function 'stream_agent_response' ready!")

Helper function 'stream_agent_response' ready!


## Test Your Implementation

Run the cells below to test your research agent with queries that use both tools.

In [12]:
# Test 1: Query that benefits from both web search AND Wikipedia
combined_query = """Research the history and current state of neural networks. I need:
1. The historical origins and key milestones (use Wikipedia for this)
2. Recent developments and breakthroughs in 2024-2025 (use web search for this)
3. Save your findings to a file called 'neural_networks_research.md'
"""

if research_agent:
    stream_agent_response(research_agent, combined_query, show_details=True, max_content_length=None)
else:
    print("Create the research agent first (complete Parts 1-5)")

Query: Research the history and current state of neural networks. I need:
1. The historical origins and key...

Streaming Agent Actions:

[USER]: Research the history and current state of neural networks. I need:
1. The historical origins and key milestones (use Wikipedia for this)
2. Recent developments and breakthroughs in 2024-2025 (use web search for this)
3. Save your findings to a file called 'neural_networks_research.md'

--------------------------------------------------------------------------------
[AGENT] Calling 1 tool(s):
  - write_todos: Planning 3 tasks
      1. [in_progress] Research the historical origins and key milestones...
      2. [pending] Search for recent developments and breakthroughs i...
      3. [pending] Synthesize findings and save to 'neural_networks_r...
--------------------------------------------------------------------------------
[TOOL: write_todos] Completed
--------------------------------------------------------------------------------
[AGENT] Ca

In [13]:
# Check if the file was created
saved_file = agent_workspace / "neural_networks_research.md"
if saved_file.exists():
    print("Research file created successfully!")
    print(f"Location: {saved_file.absolute()}")
else:
    print("Note: File may be in virtual storage if using virtual_mode=True")

Research file created successfully!
Location: d:\Agent\AI_Agents_Supplemental_Content\AI_Agents_Supplemental_Content\Level_3\research_workspace\neural_networks_research.md


In [20]:
combined_query = """Research the history and current state of Generative AI. I need:
1. The historical origins and key milestones (use Wikipedia for this)
2. Recent developments and breakthroughs in 2024-2025 (use web search for this)
3. Save your findings to a file called 'genai_research.md'
"""

if research_agent:
    stream_agent_response(research_agent, combined_query, show_details=True, max_content_length=None)
else:
    print("Create the research agent first (complete Parts 1-5)")

Query: Research the history and current state of Generative AI. I need:
1. The historical origins and key m...

Streaming Agent Actions:

[USER]: Research the history and current state of Generative AI. I need:
1. The historical origins and key milestones (use Wikipedia for this)
2. Recent developments and breakthroughs in 2024-2025 (use web search for this)
3. Save your findings to a file called 'genai_research.md'

--------------------------------------------------------------------------------
[AGENT] Calling 1 tool(s):
  - write_todos: Planning 3 tasks
      1. [in_progress] Research the historical origins and key milestones...
      2. [pending] Search for recent developments and breakthroughs i...
      3. [pending] Save the findings to a file called 'genai_research...
--------------------------------------------------------------------------------
[TOOL: write_todos] Completed
--------------------------------------------------------------------------------
[AGENT] Calling 1 tool

## Summary

In this exercise, you built a deep agent research assistant with:

1. **System Prompt**: Detailed instructions with tool usage guidelines and workflows
2. **LLM Model**: ChatOpenAI (GPT-4o) as the reasoning engine
3. **Tavily Tool**: Web search for current events and recent information
4. **Wikipedia Tool**: Encyclopedic knowledge for established facts
5. **Deep Agent Features** (provided automatically by `create_deep_agent`):
   - Planning tools (`write_todos`)
   - File system access for persistent storage
   - Subagent spawning for task decomposition

**Key Takeaways:**
- A well-crafted system prompt is essential for guiding deep agent behavior
- Deep agents combine multiple tools to handle complex research tasks
- The planning tool (`write_todos`) helps agents stay organized on multi-step tasks
- File system access enables persistent storage of research findings
- Different tools serve different purposes: web search for current info, Wikipedia for established facts